# 04 — Class-Conditional Variational Autoencoder

This notebook trains one **class-conditional VAE (CVAE)** on the normalized
`128 × 128` log-mel spectrograms created by `02_preprocess_logmel.ipynb`.

The three target classes are:

- Northern Cardinal
- Song Sparrow
- American Robin

The model is trained from randomly initialized weights; it is **not** a
fine-tuned pretrained model. Species labels condition both the encoder and the
decoder, so the model approximates

$$
p_\theta(x \mid y),
$$

where $x$ is a log-mel spectrogram and $y$ is the species label.

### Notebook responsibilities

1. Load and validate the fixed manifests and preprocessed `.npy` tensors.
2. Build a class-conditional dataset and data loaders.
3. Define the CVAE, loss, training, validation, and checkpoint logic.
4. Train with early stopping and record compute time.
5. Inspect reconstructions, conditional samples, and the latent space.
6. Save the best checkpoint, figures, generated arrays, and a run summary.

This notebook deliberately keeps all implementation in one file for the first
working version. Once stable, the reusable classes and functions can be moved
into Python modules.


## 1. Environment and dependencies

Required packages: `numpy`, `pandas`, `matplotlib`, and `torch`.

On Windows, `NUM_WORKERS = 0` is the safest Jupyter default. Mixed precision is
enabled automatically only when CUDA is available.


In [ ]:
from __future__ import annotations

import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader, Dataset

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Configuration

The default paths match the outputs previously defined for notebook `02`:

```text
processed/manifests/logmel_128_train.csv
processed/manifests/logmel_128_validation.csv
processed/manifests/logmel_128_test.csv
processed/normalization_stats.json
```

If this notebook is stored somewhere other than the project root, change only
`PROJECT_ROOT`. If the processed manifest uses a different spectrogram-path or
species column name, update the candidate lists below.

For a quick pipeline test, set `SMOKE_TEST = True`. That restricts the number of
batches and epochs; it is not a final experiment.


In [ ]:
@dataclass
class Config:
    # Paths
    PROJECT_ROOT: str = str(Path.cwd())
    PROCESSED_DIR: str = "processed"
    MANIFEST_DIR: str = "processed/manifests"
    TRAIN_MANIFEST: str = "logmel_128_train.csv"
    VAL_MANIFEST: str = "logmel_128_validation.csv"
    TEST_MANIFEST: str = "logmel_128_test.csv"
    NORMALIZATION_STATS: str = "processed/normalization_stats.json"
    PREPROCESSING_CONFIG: str = "processed/preprocessing_config.json"
    CHECKPOINT_DIR: str = "checkpoints/conditional_vae"
    OUTPUT_DIR: str = "outputs/conditional_vae"

    # Reproducibility and loading
    SEED: int = 42
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = True

    # Data contract from notebook 02
    INPUT_CHANNELS: int = 1
    IMAGE_SIZE: int = 128
    TARGET_SPECIES: Tuple[str, ...] = (
        "Northern Cardinal",
        "Song Sparrow",
        "American Robin",
    )
    SPEC_PATH_CANDIDATES: Tuple[str, ...] = (
        "relative_spec_path",
        "spec_path",
        "spectrogram_path",
        "logmel_path",
        "npy_path",
        "processed_path",
    )
    SPECIES_COLUMN_CANDIDATES: Tuple[str, ...] = (
        "name",
        "species",
        "species_name",
        "label_name",
    )
    GROUP_COLUMN_CANDIDATES: Tuple[str, ...] = (
        "id",
        "recording_id",
        "original_recording_id",
        "xc_id",
    )

    # Model
    LATENT_DIM: int = 128
    CLASS_EMBED_DIM: int = 16
    BASE_CHANNELS: int = 32

    # Optimization
    BATCH_SIZE: int = 32
    MAX_EPOCHS: int = 50
    LEARNING_RATE: float = 2e-4
    WEIGHT_DECAY: float = 1e-5
    BETA_KL: float = 1e-3
    KL_WARMUP_EPOCHS: int = 10
    GRAD_CLIP_NORM: float = 5.0
    EARLY_STOPPING_PATIENCE: int = 10
    MIN_DELTA: float = 1e-4
    AMP: bool = True

    # Execution controls
    RUN_TRAINING: bool = True
    RESUME_FROM_LAST: bool = False
    SMOKE_TEST: bool = False
    SMOKE_TEST_BATCHES: int = 3
    SMOKE_TEST_EPOCHS: int = 2
    SAMPLES_PER_SPECIES: int = 8


cfg = Config()
PROJECT_ROOT = Path(cfg.PROJECT_ROOT).resolve()
MANIFEST_DIR = PROJECT_ROOT / cfg.MANIFEST_DIR
CHECKPOINT_DIR = PROJECT_ROOT / cfg.CHECKPOINT_DIR
OUTPUT_DIR = PROJECT_ROOT / cfg.OUTPUT_DIR

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(cfg.AMP and DEVICE.type == "cuda")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Mixed precision:", AMP_ENABLED)
print("Smoke test:", cfg.SMOKE_TEST)


In [ ]:
PREPROCESSING_CONFIG_PATH = PROJECT_ROOT / cfg.PREPROCESSING_CONFIG
if not PREPROCESSING_CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing preprocessing contract: {PREPROCESSING_CONFIG_PATH}\n"
        "Run notebook 02 first, or update PREPROCESSING_CONFIG in Config."
    )

with PREPROCESSING_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    preprocessing_config = json.load(handle)

preprocessed_shape = preprocessing_config.get("output", {}).get("shape")
preprocessed_species = preprocessing_config.get("target_species")
expected_shape = [cfg.INPUT_CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE]

if preprocessed_shape != expected_shape:
    raise ValueError(
        f"Notebook 02 reports output shape {preprocessed_shape}; "
        f"this model expects {expected_shape}."
    )
if preprocessed_species != list(cfg.TARGET_SPECIES):
    raise ValueError(
        f"Notebook 02 reports species {preprocessed_species}; "
        f"this model expects {list(cfg.TARGET_SPECIES)}."
    )

print("Loaded preprocessing contract:", PREPROCESSING_CONFIG_PATH)
print("Preprocessed shape:", preprocessed_shape)
print("Preprocessed species:", preprocessed_species)


## 3. Reproducibility helpers

Deterministic settings improve repeatability, although exact GPU results can
still vary slightly across PyTorch/CUDA versions.


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(cfg.SEED)
generator = torch.Generator().manual_seed(cfg.SEED)


## 4. Load manifests and establish the label mapping

The split created in `01_dataset_audit.ipynb` must remain fixed. This section
also checks that an original recording ID does not occur in more than one split.

The notebook derives a deterministic label mapping from the selected species,
so it works even if the manifest does not already contain `label_id`.


In [ ]:
def first_existing_column(df: pd.DataFrame, candidates: Tuple[str, ...], purpose: str) -> str:
    for column in candidates:
        if column in df.columns:
            return column
    raise KeyError(
        f"Could not find a {purpose} column. Tried {list(candidates)}. "
        f"Available columns: {df.columns.tolist()}"
    )


def load_manifest(filename: str, split: str) -> pd.DataFrame:
    path = MANIFEST_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {split} manifest: {path}\n"
            "Run notebook 02 first, or update the path in Config."
        )
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"{split} manifest is empty: {path}")
    df = df.copy()
    df["split"] = split
    return df


train_df = load_manifest(cfg.TRAIN_MANIFEST, "train")
val_df = load_manifest(cfg.VAL_MANIFEST, "validation")
test_df = load_manifest(cfg.TEST_MANIFEST, "test")

SPECIES_COL = first_existing_column(
    train_df, cfg.SPECIES_COLUMN_CANDIDATES, "species"
)
SPEC_PATH_COL = first_existing_column(
    train_df, cfg.SPEC_PATH_CANDIDATES, "spectrogram path"
)

for split_name, frame in [("validation", val_df), ("test", test_df)]:
    if SPECIES_COL not in frame.columns or SPEC_PATH_COL not in frame.columns:
        raise KeyError(
            f"{split_name} manifest must contain {SPECIES_COL!r} and {SPEC_PATH_COL!r}."
        )

target_species = list(cfg.TARGET_SPECIES)
for split_name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    unexpected = sorted(set(frame[SPECIES_COL].dropna().unique()) - set(target_species))
    missing = sorted(set(target_species) - set(frame[SPECIES_COL].dropna().unique()))
    if unexpected:
        print(f"Warning: {split_name} has non-target species that will be removed: {unexpected}")
    if missing:
        raise ValueError(f"{split_name} is missing target species: {missing}")

train_df = train_df[train_df[SPECIES_COL].isin(target_species)].reset_index(drop=True)
val_df = val_df[val_df[SPECIES_COL].isin(target_species)].reset_index(drop=True)
test_df = test_df[test_df[SPECIES_COL].isin(target_species)].reset_index(drop=True)

label_to_id = {species: idx for idx, species in enumerate(target_species)}
id_to_label = {idx: species for species, idx in label_to_id.items()}

for frame in (train_df, val_df, test_df):
    frame["label_id"] = frame[SPECIES_COL].map(label_to_id).astype(int)

summary = pd.concat(
    [
        train_df.groupby(SPECIES_COL).size().rename("train"),
        val_df.groupby(SPECIES_COL).size().rename("validation"),
        test_df.groupby(SPECIES_COL).size().rename("test"),
    ],
    axis=1,
).fillna(0).astype(int).reindex(target_species)

display(summary)
print("Label mapping:", label_to_id)
print("Total clips:", int(summary.to_numpy().sum()))


In [ ]:
def find_common_group_column(frames: List[pd.DataFrame]) -> Optional[str]:
    for column in cfg.GROUP_COLUMN_CANDIDATES:
        if all(column in frame.columns for frame in frames):
            return column
    return None


GROUP_COL = find_common_group_column([train_df, val_df, test_df])
if GROUP_COL is None:
    print(
        "Warning: no original-recording ID column was found, so group leakage "
        "cannot be checked here. The split should already have been checked in notebook 01."
    )
else:
    train_groups = set(train_df[GROUP_COL].astype(str))
    val_groups = set(val_df[GROUP_COL].astype(str))
    test_groups = set(test_df[GROUP_COL].astype(str))
    overlaps = {
        "train_validation": train_groups & val_groups,
        "train_test": train_groups & test_groups,
        "validation_test": val_groups & test_groups,
    }
    assert all(len(values) == 0 for values in overlaps.values()), (
        "Original recording IDs cross split boundaries: "
        + str({key: list(value)[:5] for key, value in overlaps.items() if value})
    )
    print(f"No group leakage detected using column {GROUP_COL!r}.")


## 5. Resolve and validate spectrogram paths

Paths in a CSV may be absolute or relative. Relative paths are tested against
the project root, the processed directory, and the manifest directory. This
cell validates every file before training begins.


In [ ]:
def resolve_spec_path(path_value: str) -> Path:
    raw = Path(str(path_value))
    candidates = [raw] if raw.is_absolute() else [
        PROJECT_ROOT / raw,
        PROJECT_ROOT / cfg.PROCESSED_DIR / raw,
        MANIFEST_DIR / raw,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return candidates[0].resolve()


for frame in (train_df, val_df, test_df):
    frame["resolved_spec_path"] = frame[SPEC_PATH_COL].map(resolve_spec_path)

missing_paths = []
for split_name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    missing = frame.loc[~frame["resolved_spec_path"].map(Path.exists), "resolved_spec_path"]
    missing_paths.extend([(split_name, str(path)) for path in missing.head(10)])
    print(f"{split_name:10s}: {len(frame):5d} rows, {len(missing):3d} missing tensors")

if missing_paths:
    raise FileNotFoundError(
        "Some preprocessed tensors are missing. First examples:\n"
        + "\n".join(f"  {split}: {path}" for split, path in missing_paths)
    )

# Inspect a few files before constructing data loaders.
for path in train_df["resolved_spec_path"].head(5):
    array = np.load(path, mmap_mode="r")
    assert array.shape == (cfg.INPUT_CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE), (
        f"Unexpected shape {array.shape} in {path}"
    )
    assert array.dtype == np.float32, f"Expected float32, got {array.dtype} in {path}"
    assert np.isfinite(array).all(), f"Non-finite values found in {path}"

print("Initial tensor checks passed.")


## 6. Dataset and data loaders

Each item is returned as `(spectrogram, label_id, path)`. The path is retained
for traceability and later qualitative checks.


In [ ]:
class BirdSpectrogramDataset(Dataset):
    def __init__(self, manifest: pd.DataFrame):
        self.manifest = manifest.reset_index(drop=True).copy()

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, index: int):
        row = self.manifest.iloc[index]
        path = Path(row["resolved_spec_path"])
        array = np.load(path).astype(np.float32, copy=False)
        expected = (cfg.INPUT_CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)
        if array.shape != expected:
            raise ValueError(f"Expected {expected}, got {array.shape} in {path}")
        if not np.isfinite(array).all():
            raise ValueError(f"Non-finite values in {path}")
        x = torch.from_numpy(np.array(array, copy=True))
        y = torch.tensor(int(row["label_id"]), dtype=torch.long)
        return x, y, str(path)


train_dataset = BirdSpectrogramDataset(train_df)
val_dataset = BirdSpectrogramDataset(val_df)
test_dataset = BirdSpectrogramDataset(test_df)

loader_kwargs = dict(
    batch_size=cfg.BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=bool(cfg.PIN_MEMORY and DEVICE.type == "cuda"),
)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=generator,
    drop_last=False,
    **loader_kwargs,
)
val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **loader_kwargs)

x_batch, y_batch, path_batch = next(iter(train_loader))
print("Batch spectrograms:", tuple(x_batch.shape), x_batch.dtype)
print("Batch labels:", tuple(y_batch.shape), y_batch.dtype)
print("Value range:", float(x_batch.min()), "to", float(x_batch.max()))
print("Mean / std:", float(x_batch.mean()), float(x_batch.std()))
print("First file:", path_batch[0])


## 7. Batch sanity check

The tensors are normalized model inputs. For visualization, the notebook tries
to load the training-set normalization mean and standard deviation and converts
them back to the log-mel dB scale. If the statistics file is unavailable, it
plots normalized values and labels them accordingly.


In [ ]:
def load_normalization_stats() -> Tuple[Optional[float], Optional[float]]:
    path = PROJECT_ROOT / cfg.NORMALIZATION_STATS
    if not path.exists():
        print("Normalization statistics not found; plots will use normalized units:", path)
        return None, None
    with path.open("r", encoding="utf-8") as handle:
        stats = json.load(handle)

    mean_keys = ("mean_db", "mean", "train_mean", "global_mean", "mu_train")
    std_keys = ("std_db", "std", "train_std", "global_std", "sigma_train")
    mean = next((float(stats[key]) for key in mean_keys if key in stats), None)
    std = next((float(stats[key]) for key in std_keys if key in stats), None)
    if mean is None or std is None:
        print("Could not identify mean/std keys; plots will use normalized units.")
        return None, None
    return mean, std


TRAIN_MEAN, TRAIN_STD = load_normalization_stats()
print("Training normalization mean/std:", TRAIN_MEAN, TRAIN_STD)


def to_display_scale(x: torch.Tensor) -> np.ndarray:
    array = x.detach().cpu().float().numpy()
    if TRAIN_MEAN is not None and TRAIN_STD is not None:
        array = array * TRAIN_STD + TRAIN_MEAN
    return array


def show_spectrogram_grid(
    tensors: torch.Tensor,
    labels: torch.Tensor,
    title: str,
    max_items: int = 8,
) -> None:
    count = min(max_items, len(tensors))
    cols = min(4, count)
    rows = math.ceil(count / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.3 * rows), squeeze=False)
    for index, axis in enumerate(axes.flat):
        axis.axis("off")
        if index >= count:
            continue
        image = to_display_scale(tensors[index, 0])
        axis.imshow(image, origin="lower", aspect="auto", cmap="magma")
        axis.set_title(id_to_label[int(labels[index])])
        axis.set_xlabel("Time frame")
        axis.set_ylabel("Mel bin")
        axis.axis("on")
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()


show_spectrogram_grid(x_batch, y_batch, "Training batch sanity check")


## 8. Conditional VAE architecture

### Conditioning design

- **Encoder:** the learned species embedding is expanded spatially and
  concatenated with the input spectrogram.
- **Decoder:** the same species embedding is concatenated with the latent vector
  before upsampling.

The encoder produces $\mu$ and $\log \sigma^2$. Reparameterization uses

$$
z = \mu + \sigma \odot \epsilon,
\qquad \epsilon \sim \mathcal{N}(0, I).
$$

There is no sigmoid or tanh output activation because notebook `02` standardized
the spectrogram values rather than restricting them to `[0, 1]` or `[-1, 1]`.
Group normalization is used instead of batch normalization so the model remains
stable with modest batch sizes.


In [ ]:
def group_count(channels: int, maximum: int = 8) -> int:
    for groups in range(min(maximum, channels), 0, -1):
        if channels % groups == 0:
            return groups
    return 1


class DownBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(group_count(out_channels), out_channels),
            nn.SiLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels, out_channels, kernel_size=4, stride=2, padding=1
            ),
            nn.GroupNorm(group_count(out_channels), out_channels),
            nn.SiLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ConditionalVAE(nn.Module):
    def __init__(
        self,
        num_classes: int,
        image_size: int = 128,
        input_channels: int = 1,
        latent_dim: int = 128,
        class_embed_dim: int = 16,
        base_channels: int = 32,
    ):
        super().__init__()
        if image_size != 128:
            raise ValueError("This first architecture expects 128×128 inputs.")

        self.num_classes = num_classes
        self.image_size = image_size
        self.input_channels = input_channels
        self.latent_dim = latent_dim
        self.class_embed_dim = class_embed_dim
        self.base_channels = base_channels

        self.class_embedding = nn.Embedding(num_classes, class_embed_dim)

        b = base_channels
        self.encoder = nn.Sequential(
            DownBlock(input_channels + class_embed_dim, b),       # 128 -> 64
            DownBlock(b, b * 2),                                  # 64 -> 32
            DownBlock(b * 2, b * 4),                              # 32 -> 16
            DownBlock(b * 4, b * 8),                              # 16 -> 8
            DownBlock(b * 8, b * 8),                              # 8 -> 4
        )
        self.encoder_flat_dim = b * 8 * 4 * 4
        self.fc_mu = nn.Linear(self.encoder_flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.encoder_flat_dim, latent_dim)

        self.decoder_input = nn.Linear(latent_dim + class_embed_dim, self.encoder_flat_dim)
        self.decoder = nn.Sequential(
            UpBlock(b * 8, b * 8),                                # 4 -> 8
            UpBlock(b * 8, b * 4),                                # 8 -> 16
            UpBlock(b * 4, b * 2),                                # 16 -> 32
            UpBlock(b * 2, b),                                    # 32 -> 64
            UpBlock(b, max(b // 2, 8)),                            # 64 -> 128
            nn.Conv2d(max(b // 2, 8), input_channels, kernel_size=3, padding=1),
        )

    def encode(self, x: torch.Tensor, labels: torch.Tensor):
        class_embed = self.class_embedding(labels)
        condition_map = class_embed[:, :, None, None].expand(
            -1, -1, self.image_size, self.image_size
        )
        conditioned_x = torch.cat([x, condition_map], dim=1)
        features = self.encoder(conditioned_x).flatten(start_dim=1)
        return self.fc_mu(features), self.fc_logvar(features)

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        return mu + epsilon * std

    def decode(self, z: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        class_embed = self.class_embedding(labels)
        hidden = self.decoder_input(torch.cat([z, class_embed], dim=1))
        hidden = hidden.view(-1, self.base_channels * 8, 4, 4)
        return self.decoder(hidden)

    def forward(self, x: torch.Tensor, labels: torch.Tensor):
        mu, logvar = self.encode(x, labels)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z, labels)
        return reconstruction, mu, logvar

    @torch.no_grad()
    def sample(
        self,
        labels: torch.Tensor,
        generator: Optional[torch.Generator] = None,
    ) -> torch.Tensor:
        z = torch.randn(
            len(labels),
            self.latent_dim,
            device=labels.device,
            generator=generator,
        )
        return self.decode(z, labels)


model = ConditionalVAE(
    num_classes=len(label_to_id),
    image_size=cfg.IMAGE_SIZE,
    input_channels=cfg.INPUT_CHANNELS,
    latent_dim=cfg.LATENT_DIM,
    class_embed_dim=cfg.CLASS_EMBED_DIM,
    base_channels=cfg.BASE_CHANNELS,
).to(DEVICE)

with torch.no_grad():
    sample_x = x_batch[:2].to(DEVICE)
    sample_y = y_batch[:2].to(DEVICE)
    sample_recon, sample_mu, sample_logvar = model(sample_x, sample_y)

assert sample_recon.shape == sample_x.shape
assert sample_mu.shape == (2, cfg.LATENT_DIM)
assert sample_logvar.shape == (2, cfg.LATENT_DIM)
print("Forward-pass shape check passed:", tuple(sample_recon.shape))


## 9. Model size and a quick memory check

The parameter count is reported to address the compute-resource concern. Actual peak memory is measured during training when CUDA is
available.


In [ ]:
def count_parameters(module: nn.Module) -> Dict[str, int]:
    total = sum(parameter.numel() for parameter in module.parameters())
    trainable = sum(
        parameter.numel() for parameter in module.parameters() if parameter.requires_grad
    )
    return {"total": total, "trainable": trainable}


parameter_counts = count_parameters(model)
parameter_memory_mb = parameter_counts["total"] * 4 / (1024 ** 2)

print(f"Total parameters:     {parameter_counts['total']:,}")
print(f"Trainable parameters: {parameter_counts['trainable']:,}")
print(f"FP32 parameter memory only: {parameter_memory_mb:.1f} MB")

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


## 10. VAE objective

The total loss is

$$
\mathcal{L}
=
\mathcal{L}_{\mathrm{recon}}
+
\beta\,\mathcal{L}_{\mathrm{KL}},
$$

with mean-squared reconstruction error and

$$
\mathcal{L}_{\mathrm{KL}}
=
-\frac{1}{2}
\mathbb{E}\left[
1 + \log \sigma^2 - \mu^2 - \sigma^2
\right].
$$

Both terms are averaged so their scale is less sensitive to batch size and image
resolution. A linear KL warm-up reduces early posterior collapse: $\beta$ rises
from zero to `BETA_KL` during the first `KL_WARMUP_EPOCHS`.


In [ ]:
def vae_loss(
    reconstruction: torch.Tensor,
    target: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    beta: float,
) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    recon_loss = F.mse_loss(reconstruction, target, reduction="mean")
    kl_loss = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kl_loss
    return total_loss, {"loss": total_loss, "recon": recon_loss, "kl": kl_loss}


def beta_for_epoch(epoch: int) -> float:
    if cfg.KL_WARMUP_EPOCHS <= 0:
        return cfg.BETA_KL
    fraction = min(1.0, epoch / cfg.KL_WARMUP_EPOCHS)
    return cfg.BETA_KL * fraction


test_loss, test_parts = vae_loss(
    sample_recon, sample_x, sample_mu, sample_logvar, beta=cfg.BETA_KL
)
print({key: float(value) for key, value in test_parts.items()})


## 11. Training and validation utilities

Features included here:

- optional CUDA mixed precision;
- gradient clipping;
- validation after every epoch;
- best and last checkpoints;
- optional resume from the last checkpoint;
- early stopping;
- per-epoch elapsed time and peak CUDA memory.


In [ ]:
def make_grad_scaler(enabled: bool):
    # Supports both newer and older PyTorch AMP APIs.
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_context(enabled: bool):
    try:
        return torch.amp.autocast(device_type=DEVICE.type, enabled=enabled)
    except AttributeError:
        return torch.cuda.amp.autocast(enabled=enabled)


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    beta: float,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scaler=None,
    max_batches: Optional[int] = None,
) -> Dict[str, float]:
    is_training = optimizer is not None
    model.train(is_training)

    totals = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
    sample_count = 0

    for batch_index, (x, labels, _) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break

        x = x.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        batch_size = len(x)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            with autocast_context(AMP_ENABLED):
                reconstruction, mu, logvar = model(x, labels)
                loss, parts = vae_loss(reconstruction, x, mu, logvar, beta)

            if is_training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()

        for key in totals:
            totals[key] += float(parts[key].detach()) * batch_size
        sample_count += batch_size

    if sample_count == 0:
        raise RuntimeError("No samples were processed in this epoch.")
    return {key: value / sample_count for key, value in totals.items()}


def save_checkpoint(
    path: Path,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler,
    history: Dict[str, List[float]],
    best_val_loss: float,
) -> None:
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "best_val_loss": best_val_loss,
        "config": asdict(cfg),
        "label_to_id": label_to_id,
        "parameter_counts": parameter_counts,
    }
    torch.save(payload, path)


def load_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scaler=None,
):
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scaler is not None and "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    return checkpoint


## 12. Train the model

The default experiment trains for at most 50 epochs, with early stopping. Before
the final run, execute once with `SMOKE_TEST = True` to confirm that the complete
pipeline and checkpoint logic work.

Training loss is **not** directly comparable to the later diffusion objective;
the fair comparison is based on held-out sample quality and shared evaluation
metrics in notebook `06`.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY,
)
scaler = make_grad_scaler(AMP_ENABLED)

history = {
    "epoch": [],
    "beta": [],
    "train_loss": [],
    "train_recon": [],
    "train_kl": [],
    "val_loss": [],
    "val_recon": [],
    "val_kl": [],
    "epoch_seconds": [],
}

best_path = CHECKPOINT_DIR / "conditional_vae_best.pt"
last_path = CHECKPOINT_DIR / "conditional_vae_last.pt"
start_epoch = 1
best_val_loss = float("inf")
epochs_without_improvement = 0

if cfg.RESUME_FROM_LAST and last_path.exists():
    checkpoint = load_checkpoint(last_path, model, optimizer, scaler)
    start_epoch = int(checkpoint["epoch"]) + 1
    history = checkpoint.get("history", history)
    best_val_loss = float(checkpoint.get("best_val_loss", best_val_loss))
    print(f"Resuming from epoch {start_epoch} using {last_path}")

max_epochs = cfg.SMOKE_TEST_EPOCHS if cfg.SMOKE_TEST else cfg.MAX_EPOCHS
max_batches = cfg.SMOKE_TEST_BATCHES if cfg.SMOKE_TEST else None
training_start = time.perf_counter()

if cfg.RUN_TRAINING:
    for epoch in range(start_epoch, max_epochs + 1):
        epoch_start = time.perf_counter()
        beta = beta_for_epoch(epoch)

        train_metrics = run_epoch(
            model,
            train_loader,
            beta=beta,
            optimizer=optimizer,
            scaler=scaler,
            max_batches=max_batches,
        )
        val_metrics = run_epoch(
            model,
            val_loader,
            beta=beta,
            optimizer=None,
            scaler=None,
            max_batches=max_batches,
        )

        elapsed = time.perf_counter() - epoch_start
        history["epoch"].append(epoch)
        history["beta"].append(beta)
        history["train_loss"].append(train_metrics["loss"])
        history["train_recon"].append(train_metrics["recon"])
        history["train_kl"].append(train_metrics["kl"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_recon"].append(val_metrics["recon"])
        history["val_kl"].append(val_metrics["kl"])
        history["epoch_seconds"].append(elapsed)

        improved = val_metrics["loss"] < best_val_loss - cfg.MIN_DELTA
        if improved:
            best_val_loss = val_metrics["loss"]
            epochs_without_improvement = 0
            save_checkpoint(
                best_path, epoch, model, optimizer, scaler, history, best_val_loss
            )
        else:
            epochs_without_improvement += 1

        save_checkpoint(
            last_path, epoch, model, optimizer, scaler, history, best_val_loss
        )

        print(
            f"Epoch {epoch:03d}/{max_epochs:03d} | beta={beta:.5f} | "
            f"train={train_metrics['loss']:.5f} "
            f"(recon={train_metrics['recon']:.5f}, kl={train_metrics['kl']:.5f}) | "
            f"val={val_metrics['loss']:.5f} "
            f"(recon={val_metrics['recon']:.5f}, kl={val_metrics['kl']:.5f}) | "
            f"{elapsed:.1f}s" + (" | saved best" if improved else "")
        )

        if epochs_without_improvement >= cfg.EARLY_STOPPING_PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break

    total_training_seconds = time.perf_counter() - training_start
    print(f"Training section elapsed: {total_training_seconds / 60:.1f} minutes")
else:
    total_training_seconds = 0.0
    print("RUN_TRAINING is False; no optimization was performed.")

if best_path.exists():
    best_checkpoint = load_checkpoint(best_path, model)
    history = best_checkpoint.get("history", history)
    print("Loaded best checkpoint from epoch", best_checkpoint["epoch"])
else:
    print("No best checkpoint exists yet. Run training before final evaluation.")

if DEVICE.type == "cuda":
    peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"Peak allocated CUDA memory: {peak_memory_gb:.2f} GB")
else:
    peak_memory_gb = None


## 13. Training curves

Inspect reconstruction and KL terms separately. A KL value that collapses close
to zero may indicate that the decoder is ignoring the latent variable. If that
happens, reduce `BETA_KL`, lengthen warm-up, or reduce decoder capacity.


In [ ]:
def plot_training_history(history: Dict[str, List[float]]) -> None:
    if not history["epoch"]:
        print("No history to plot.")
        return

    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="validation")
    axes[0].set_title("Total VAE loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["train_recon"], label="train")
    axes[1].plot(epochs, history["val_recon"], label="validation")
    axes[1].set_title("Reconstruction MSE")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(epochs, history["train_kl"], label="train")
    axes[2].plot(epochs, history["val_kl"], label="validation")
    axes[2].set_title("KL term")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    for axis in axes:
        axis.grid(alpha=0.25)
    fig.tight_layout()
    figure_path = OUTPUT_DIR / "training_curves.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)


plot_training_history(history)


## 14. Held-out test reconstruction metrics

Reconstruction is meaningful for the VAE, but it is not the only measure of
generation quality. Notebook `06` will compare freely generated VAE and
diffusion samples using shared metrics.


In [ ]:
if best_path.exists():
    final_beta = cfg.BETA_KL
    test_metrics = run_epoch(
        model,
        test_loader,
        beta=final_beta,
        optimizer=None,
        scaler=None,
        max_batches=cfg.SMOKE_TEST_BATCHES if cfg.SMOKE_TEST else None,
    )
    print("Held-out test metrics:")
    for key, value in test_metrics.items():
        print(f"  {key:8s}: {value:.6f}")
else:
    test_metrics = {}
    print("Train the model first to compute meaningful test metrics.")


## 15. Reconstruction examples by species

The comparison below uses unseen test inputs and displays the encoder mean
$\mu$ rather than a random posterior sample, making the reconstruction panel
repeatable.


In [ ]:
def collect_one_per_species(dataset: BirdSpectrogramDataset):
    found = {}
    for index in range(len(dataset)):
        x, label, path = dataset[index]
        label_id = int(label)
        if label_id not in found:
            found[label_id] = (x, label, path)
        if len(found) == len(label_to_id):
            break
    return [found[index] for index in range(len(label_to_id))]


@torch.no_grad()
def plot_reconstructions(model: ConditionalVAE) -> None:
    examples = collect_one_per_species(test_dataset)
    x = torch.stack([item[0] for item in examples]).to(DEVICE)
    labels = torch.stack([item[1] for item in examples]).to(DEVICE)

    model.eval()
    mu, _ = model.encode(x, labels)
    reconstructions = model.decode(mu, labels)

    fig, axes = plt.subplots(len(examples), 2, figsize=(10, 3.5 * len(examples)))
    for row in range(len(examples)):
        original = to_display_scale(x[row, 0])
        reconstructed = to_display_scale(reconstructions[row, 0])
        common_min = min(float(original.min()), float(reconstructed.min()))
        common_max = max(float(original.max()), float(reconstructed.max()))

        axes[row, 0].imshow(
            original, origin="lower", aspect="auto", cmap="magma",
            vmin=common_min, vmax=common_max,
        )
        axes[row, 1].imshow(
            reconstructed, origin="lower", aspect="auto", cmap="magma",
            vmin=common_min, vmax=common_max,
        )
        species = id_to_label[int(labels[row])]
        axes[row, 0].set_title(f"Original — {species}")
        axes[row, 1].set_title(f"Reconstruction — {species}")
        for axis in axes[row]:
            axis.set_xlabel("Time frame")
            axis.set_ylabel("Mel bin")

    fig.tight_layout()
    figure_path = OUTPUT_DIR / "test_reconstructions.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)


if best_path.exists():
    plot_reconstructions(model)
else:
    print("Train the model first.")


## 16. Conditional generation by species

For each species, sample $z \sim \mathcal{N}(0, I)$ and decode with the requested
label. The generated arrays are saved in the **same normalized tensor space** as
the model inputs, with shape `(1, 128, 128)`. Notebook `06` can therefore load
them directly for classifier recognizability, embedding-distance, diversity, and
nearest-neighbor analyses.


In [ ]:
@torch.no_grad()
def generate_and_save_samples(
    model: ConditionalVAE,
    samples_per_species: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    labels = torch.arange(len(label_to_id), device=DEVICE).repeat_interleave(
        samples_per_species
    )
    samples = model.sample(labels)

    generated_dir = OUTPUT_DIR / "generated_npy"
    generated_dir.mkdir(parents=True, exist_ok=True)
    records = []
    for index, (sample, label) in enumerate(zip(samples, labels)):
        label_id = int(label)
        species = id_to_label[label_id]
        species_slug = species.lower().replace(" ", "_").replace("'", "")
        species_index = index % samples_per_species
        filename = f"vae_{species_slug}_{species_index:03d}.npy"
        path = generated_dir / filename
        np.save(path, sample.detach().cpu().float().numpy().astype(np.float32))
        records.append(
            {
                "filename": filename,
                "spec_path": str(path.resolve()),
                "name": species,
                "label_id": label_id,
                "source_model": "conditional_vae",
            }
        )

    generated_manifest = pd.DataFrame(records)
    manifest_path = OUTPUT_DIR / "generated_manifest.csv"
    generated_manifest.to_csv(manifest_path, index=False)
    print("Saved generated manifest:", manifest_path)
    return samples.cpu(), labels.cpu()


def plot_generated_samples(
    samples: torch.Tensor,
    labels: torch.Tensor,
    samples_per_species_to_show: int = 4,
) -> None:
    rows = len(label_to_id)
    cols = samples_per_species_to_show
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows), squeeze=False)

    for label_id in range(rows):
        indices = torch.where(labels == label_id)[0][:cols]
        for col, index in enumerate(indices):
            image = to_display_scale(samples[index, 0])
            axes[label_id, col].imshow(image, origin="lower", aspect="auto", cmap="magma")
            axes[label_id, col].set_title(f"{id_to_label[label_id]} — {col + 1}")
            axes[label_id, col].set_xlabel("Time frame")
            axes[label_id, col].set_ylabel("Mel bin")

    fig.tight_layout()
    figure_path = OUTPUT_DIR / "conditional_samples.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)


if best_path.exists():
    generated_samples, generated_labels = generate_and_save_samples(
        model, cfg.SAMPLES_PER_SPECIES
    )
    plot_generated_samples(generated_samples, generated_labels)
else:
    print("Train the model first.")


## 17. Latent-space inspection

This diagnostic encodes the held-out test set using $\mu$ and projects the
latent vectors to two dimensions with an SVD-based PCA implementation. Clear
class separation is not required, but the plot helps reveal whether species
information and meaningful variation are represented in latent space.


In [ ]:
@torch.no_grad()
def collect_latents(
    model: ConditionalVAE,
    loader: DataLoader,
    max_samples: int = 1000,
) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    latent_batches = []
    label_batches = []
    collected = 0
    for x, labels, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        mu, _ = model.encode(x, labels)
        latent_batches.append(mu.cpu().numpy())
        label_batches.append(labels.cpu().numpy())
        collected += len(x)
        if collected >= max_samples:
            break
    return (
        np.concatenate(latent_batches, axis=0)[:max_samples],
        np.concatenate(label_batches, axis=0)[:max_samples],
    )


def pca_2d(x: np.ndarray) -> np.ndarray:
    centered = x - x.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ vt[:2].T


if best_path.exists():
    latent_vectors, latent_labels = collect_latents(model, test_loader)
    projected = pca_2d(latent_vectors)

    fig, axis = plt.subplots(figsize=(8, 6))
    for label_id, species in id_to_label.items():
        mask = latent_labels == label_id
        axis.scatter(
            projected[mask, 0],
            projected[mask, 1],
            s=20,
            alpha=0.7,
            label=species,
        )
    axis.set_title("Held-out test latent means — PCA projection")
    axis.set_xlabel("PC 1")
    axis.set_ylabel("PC 2")
    axis.grid(alpha=0.2)
    axis.legend()
    fig.tight_layout()
    figure_path = OUTPUT_DIR / "latent_pca.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", figure_path)
else:
    print("Train the model first.")


## 18. Save run metadata and final verification

The summary records the model scale, core hyperparameters, timing, peak memory,
best validation loss, and held-out test reconstruction metrics. This information
can be reused in the report and in the VAE-vs-diffusion comparison.


In [ ]:
completed_epochs = len(history["epoch"])
best_epoch = None
if history["val_loss"]:
    best_index = int(np.argmin(history["val_loss"]))
    best_epoch = int(history["epoch"][best_index])

run_summary = {
    "model": "class_conditional_vae",
    "trained_from_scratch": True,
    "target_species": list(cfg.TARGET_SPECIES),
    "label_to_id": label_to_id,
    "input_shape": [cfg.INPUT_CHANNELS, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE],
    "latent_dim": cfg.LATENT_DIM,
    "class_embed_dim": cfg.CLASS_EMBED_DIM,
    "base_channels": cfg.BASE_CHANNELS,
    "total_parameters": parameter_counts["total"],
    "trainable_parameters": parameter_counts["trainable"],
    "batch_size": cfg.BATCH_SIZE,
    "learning_rate": cfg.LEARNING_RATE,
    "beta_kl": cfg.BETA_KL,
    "kl_warmup_epochs": cfg.KL_WARMUP_EPOCHS,
    "completed_epochs": completed_epochs,
    "best_epoch": best_epoch,
    "best_validation_loss": (
        float(min(history["val_loss"])) if history["val_loss"] else None
    ),
    "test_metrics": {key: float(value) for key, value in test_metrics.items()},
    "training_section_seconds": float(total_training_seconds),
    "mean_epoch_seconds": (
        float(np.mean(history["epoch_seconds"])) if history["epoch_seconds"] else None
    ),
    "peak_allocated_cuda_gb": peak_memory_gb,
    "device": str(DEVICE),
    "smoke_test": cfg.SMOKE_TEST,
    "best_checkpoint": str(best_path.resolve()) if best_path.exists() else None,
}

summary_path = OUTPUT_DIR / "run_summary.json"
with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(run_summary, handle, indent=2, ensure_ascii=False)

print(json.dumps(run_summary, indent=2, ensure_ascii=False))
print("Saved run summary:", summary_path)

if best_path.exists():
    assert best_path.stat().st_size > 0
    generated_manifest_path = OUTPUT_DIR / "generated_manifest.csv"
    assert generated_manifest_path.exists(), "Generated manifest is missing."
    generated_df = pd.read_csv(generated_manifest_path)
    expected_generated = len(label_to_id) * cfg.SAMPLES_PER_SPECIES
    assert len(generated_df) == expected_generated
    assert generated_df["spec_path"].map(lambda p: Path(p).exists()).all()
    print("Final artifact checks passed.")


## 19. Interpretation checklist

After the final run, record short observations here before moving to notebook
`05_conditional_diffusion.ipynb`:

- Did train and validation losses converge without a large generalization gap?
- Did the KL term remain non-zero, or did posterior collapse occur?
- Which species produced the clearest reconstruction structure?
- Are samples within each species diverse, or visually repetitive?
- Do condition labels visibly affect generated time-frequency patterns?
- What were the model size, mean epoch time, total training time, and peak GPU
  memory?
- Which VAE limitations should the diffusion model be expected to improve?

Do not judge the VAE only from reconstruction loss. Final claims should combine
the shared evaluation metrics from notebook `06` with visual inspection and a
small number of audio demonstrations.
